### Phase 2 — Descriptive & Diagnostic Analytics

#### Load Data & Build Delivered-Orders Dataset

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

products    = pd.read_csv('data/olist_products_dataset.csv')
order_items = pd.read_csv('data/olist_order_items_dataset.csv')
orders      = pd.read_csv('data/olist_orders_dataset.csv')
customers   = pd.read_csv('data/olist_customers_dataset.csv')
reviews     = pd.read_csv('data/olist_order_reviews_dataset.csv')

hw_products  = products[products['product_category_name'] == 'utilidades_domesticas'].copy()
hw_items     = order_items[order_items['product_id'].isin(hw_products['product_id'])].copy()
hw_order_ids = hw_items['order_id'].unique()
hw_orders    = orders[orders['order_id'].isin(hw_order_ids)].copy()

hw_orders = hw_orders.merge(
    customers[['customer_id', 'customer_unique_id', 'customer_state']], on='customer_id', how='left'
)
hw_orders = hw_orders.merge(
    reviews[['order_id', 'review_score']].drop_duplicates('order_id'), on='order_id', how='left'
)

items_rev = hw_items.groupby('order_id').agg(
    total_price   = ('price', 'sum'),
    total_freight = ('freight_value', 'sum')
).reset_index()
hw_orders = hw_orders.merge(items_rev, on='order_id', how='left')

for col in ['order_purchase_timestamp', 'order_approved_at',
            'order_delivered_carrier_date', 'order_delivered_customer_date',
            'order_estimated_delivery_date']:
    hw_orders[col] = pd.to_datetime(hw_orders[col])

delivered = hw_orders[hw_orders['order_status'] == 'delivered'].copy()

delivered['delivery_days'] = (
    delivered['order_delivered_customer_date'] - delivered['order_purchase_timestamp']
).dt.total_seconds() / 86400
delivered['seller_days'] = (
    delivered['order_delivered_carrier_date'] - delivered['order_purchase_timestamp']
).dt.total_seconds() / 86400
delivered['carrier_days'] = (
    delivered['order_delivered_customer_date'] - delivered['order_delivered_carrier_date']
).dt.total_seconds() / 86400
delivered['is_late']   = delivered['order_delivered_customer_date'] > delivered['order_estimated_delivery_date']
delivered['month_str'] = delivered['order_purchase_timestamp'].dt.strftime('%Y-%m')

print("=" * 40)
print(f"Delivered Houseware orders ready: {len(delivered):,}")
print("=" * 40)

Delivered Houseware orders ready: 5,743


#### Overall Housewares KPIs

In [2]:
total_rev   = delivered['total_price'].sum()
avg_deliv   = delivered['delivery_days'].mean()
avg_seller  = delivered['seller_days'].mean()
avg_carrier = delivered['carrier_days'].mean()
on_time_pct = (1 - delivered['is_late'].mean()) * 100
avg_review  = delivered['review_score'].mean()

# Repeat Customer Rate calculation
cust_order_counts = delivered.groupby('customer_unique_id')['order_id'].nunique()
total_customers   = len(cust_order_counts)
repeat_customers  = (cust_order_counts > 1).sum()
repeat_rate       = (repeat_customers / total_customers) * 100

print("=" * 48)
print("  HOUSEWARES KPIs  (Delivered Orders Only)")
print("=" * 48)
print(f"Total Category Revenue          : R$ {total_rev:,.2f}")
print(f"Total Delivered Orders          : {len(delivered):,}")
print(f"Total Unique Customers          : {total_customers:,}")
print(f"Repeat Customer Rate (Category) : {repeat_rate:.2f}%  ({repeat_customers} repeat buyers)")
print(f"Avg Delivery Lead Time          : {avg_deliv:.2f} days")
print(f"   - Seller Dispatch Leg        : {avg_seller:.2f} days")
print(f"   - Carrier Transit Leg        : {avg_carrier:.2f} days")
print(f"On-Time Delivery Rate           : {on_time_pct:.2f}%  (Target: >= 95%)")
print(f"Avg Customer Review Score       : {avg_review:.2f} / 5.0")

  HOUSEWARES KPIs  (Delivered Orders Only)
Total Category Revenue          : R$ 615,628.69
Total Delivered Orders          : 5,743
Total Unique Customers          : 5,681
Repeat Customer Rate (Category) : 1.06%  (60 repeat buyers)
Avg Delivery Lead Time          : 11.09 days
   - Seller Dispatch Leg        : 3.06 days
   - Carrier Transit Leg        : 8.03 days
On-Time Delivery Rate           : 93.05%  (Target: >= 95%)
Avg Customer Review Score       : 4.19 / 5.0


#### Monthly Revenue & Delivery Performance (2017-2018)

In [3]:
monthly = (
    delivered[delivered['month_str'] >= '2017-01']
    .groupby('month_str')
    .agg(
        total_sales  = ('total_price',   'sum'),
        order_count  = ('order_id',      'count'),
        avg_delivery = ('delivery_days', 'mean'),
        avg_seller   = ('seller_days',   'mean'),
        avg_carrier  = ('carrier_days',  'mean'),
        late_rate    = ('is_late',       'mean'),
        avg_review   = ('review_score',  'mean')
    )
    .reset_index()
)
monthly['late_pct'] = (monthly['late_rate'] * 100).round(2)

display = monthly[['month_str','order_count','total_sales','avg_delivery',
                   'avg_seller','avg_carrier','late_pct','avg_review']].copy()
display.columns = ['Month','Orders','Sales (R$)','Avg Deliv (d)',
                   'Seller Leg (d)','Carrier Leg (d)','Late %','Avg Review']
display = display.set_index('Month').round(2)
print(display.to_string())

         Orders  Sales (R$)  Avg Deliv (d)  Seller Leg (d)  Carrier Leg (d)  Late %  Avg Review
Month                                                                                          
2017-01      21     2827.58          12.41            5.59             6.82    4.76        4.33
2017-02      61    11295.34          14.55            3.23            11.32    3.28        4.10
2017-03     170    12906.96          13.19            3.37             9.83    7.06        4.22
2017-04     134    12472.18          14.67            4.10            10.58    7.46        4.10
2017-05     259    21374.90          10.20            2.91             7.29    1.16        4.24
2017-06     280    22978.21          11.27            3.11             8.16    3.57        4.25
2017-07     220    20251.53          10.72            3.48             7.23    2.73        4.31
2017-08     245    23427.84           9.73            2.87             6.86    2.45        4.21
2017-09     217    19941.56          10.

#### Top 5 States by Revenue

In [4]:
top5 = (
    delivered.groupby('customer_state')
    .agg(
        revenue      = ('total_price',   'sum'),
        orders       = ('order_id',      'count'),
        avg_delivery = ('delivery_days', 'mean'),
        late_rate    = ('is_late',       'mean'),
        avg_review   = ('review_score',  'mean')
    )
    .sort_values('revenue', ascending=False)
    .head(5)
    .reset_index()
)
top5['late_pct'] = (top5['late_rate'] * 100).round(2)
display = top5[['customer_state','revenue','orders','avg_delivery','late_pct','avg_review']].copy()
display.columns = ['State','Revenue (R$)','Orders','Avg Deliv (days)','Late %','Avg Review']
print(display.set_index('State').round(2).to_string())

       Revenue (R$)  Orders  Avg Deliv (days)  Late %  Avg Review
State                                                            
SP        269656.60    2720              8.07    6.03        4.27
MG         74910.80     685             11.14    4.53        4.19
RJ         74080.42     717             13.72    9.34        4.04
RS         38365.95     345             14.32    8.12        4.14
PR         31162.61     282             11.63    3.90        4.16


#### Geographic Breakdown — All States Performance

In [5]:
all_states = (
    delivered.groupby('customer_state')
    .agg(
        revenue      = ('total_price',   'sum'),
        orders       = ('order_id',      'count'),
        avg_delivery = ('delivery_days', 'mean'),
        late_rate    = ('is_late',       'mean'),
        avg_review   = ('review_score',  'mean')
    )
    .sort_values('revenue', ascending=False)
    .reset_index()
)
all_states['late_pct'] = (all_states['late_rate'] * 100).round(2)
all_display = all_states[['customer_state','revenue','orders','avg_delivery','late_pct','avg_review']].copy()
all_display.columns = ['State','Revenue (R$)','Orders','Avg Deliv (days)','Late %','Avg Review']
print(all_display.set_index('State').round(2).to_string())

       Revenue (R$)  Orders  Avg Deliv (days)  Late %  Avg Review
State                                                            
SP        269656.60    2720              8.07    6.03        4.27
MG         74910.80     685             11.14    4.53        4.19
RJ         74080.42     717             13.72    9.34        4.04
RS         38365.95     345             14.32    8.12        4.14
PR         31162.61     282             11.63    3.90        4.16
SC         26674.56     218             14.08   11.47        4.09
BA         16585.98     136             17.05    8.82        4.01
GO         14733.30      98             14.43    3.06        4.21
DF         12842.46     121             11.50    3.31        4.23
ES         11509.46     101             16.59   14.85        4.13
MS         10861.74      34             15.64   11.76        4.18
CE          6958.06      44             20.07   13.64        4.25
PE          5101.72      53             15.83    1.89        4.40
PA        

#### Identifying the Outlier Month

In [6]:
worst = monthly.loc[monthly['late_pct'].idxmax()]
print("=" * 48)
print("            WORST PERFORMING MONTH")
print("=" * 48)
print(f"Month              : {worst['month_str']}")
print(f"Orders             : {int(worst['order_count'])}")
print(f"Late Delivery Rate : {worst['late_pct']:.2f}%  <- HIGHEST IN CATEGORY HISTORY")
print(f"Avg Delivery Days  : {worst['avg_delivery']:.2f}")
print(f"Seller Leg         : {worst['avg_seller']:.2f} days  (category avg: {avg_seller:.2f} days)")
print(f"Carrier Leg        : {worst['avg_carrier']:.2f} days  (category avg: {avg_carrier:.2f} days)")
print(f"Avg Review Score   : {worst['avg_review']:.2f} / 5.0  <- LOWEST IN CATEGORY HISTORY")

            WORST PERFORMING MONTH
Month              : 2018-03
Orders             : 322
Late Delivery Rate : 18.01%  <- HIGHEST IN CATEGORY HISTORY
Avg Delivery Days  : 14.74
Seller Leg         : 2.84 days  (category avg: 3.06 days)
Carrier Leg        : 11.90 days  (category avg: 8.03 days)
Avg Review Score   : 3.86 / 5.0  <- LOWEST IN CATEGORY HISTORY


#### Customer Satisfaction Impact of Late Deliveries

In [7]:
review_split = delivered.groupby('is_late')['review_score'].agg(['mean','count'])
review_split.index = ['On-Time', 'Late']
review_split.columns = ['Avg Review Score', 'Order Count']
print(review_split.round(2).to_string())
penalty = review_split.loc['On-Time','Avg Review Score'] - review_split.loc['Late','Avg Review Score']
print("\n")
print("=" * 60)
print(f"Customer satisfaction PENALTY for late delivery: -{penalty:.2f} stars")
print("=" * 60)

         Avg Review Score  Order Count
On-Time              4.30         5319
Late                 2.71          390


Customer satisfaction PENALTY for late delivery: -1.60 stars
